## # Système d'alerte précoce — Risque d'inondation à Libreville

Ce notebook combine :
1. Le modèle prédictif entraîné sur les données météorologiques (notebook 1),
2. L'indice de risque géospatial par quartier (notebook 2),

afin de générer une alerte de risque d'inondation, différenciée par quartier,
à partir des conditions météorologiques du jour.

## ## Étape 1 : Chargement du modèle et des données nécessaires

In [2]:
import pandas as pd
import joblib

# Chargement du modèle entraîné et sauvegardé dans le notebook 1
modele = joblib.load(r"C:\Users\ADS\Desktop\Projet_Inondation\modele_random_forest_inondation.pkl")

# Chargement des données météo (pour récupérer les dernières valeurs connues)
df_meteo = pd.read_csv(
    r"C:\Users\ADS\Desktop\Projet_Inondation\donnees_meteo_libreville.csv",
    parse_dates=["date"]
)

print("Modèle chargé avec succès ✅")
print(f"Dernière date disponible dans les données météo : {df_meteo['date'].max().date()}")

Modèle chargé avec succès ✅
Dernière date disponible dans les données météo : 2026-07-07


## ## Étape 2 : Calcul de la probabilité de risque météorologique du jour

Plutôt qu'une simple réponse binaire (oui/non), on utilise la probabilité
calculée par le modèle : cela permet de distinguer un risque faible d'un
risque élevé, et donc de définir des niveaux d'alerte gradués.

In [3]:
# Liste des variables utilisées par le modèle (dans le même ordre que l'entraînement)
colonnes_features = [
    "precipitation_sum",
    "temperature_2m_max",
    "temperature_2m_min",
    "temperature_2m_mean",
    "relative_humidity_2m_mean",
    "wind_speed_10m_max",
    "surface_pressure_mean",
    "pluie_cumul_3j",
    "pluie_cumul_7j",
    "pluie_cumul_15j",
    "humidite_moyenne_7j",
    "amplitude_thermique",
]

# On récupère les données du jour le plus récent disponible
donnees_du_jour = df_meteo.sort_values("date").iloc[[-1]]
date_du_jour = donnees_du_jour["date"].values[0]

# Extraction des variables nécessaires pour la prédiction
X_jour = donnees_du_jour[colonnes_features]

# predict_proba() renvoie deux probabilités : [proba classe 0, proba classe 1]
# On garde uniquement la probabilité de la classe 1 (inondation)
probabilite_inondation = modele.predict_proba(X_jour)[0][1]

print(f"Date analysée : {pd.to_datetime(date_du_jour).date()}")
print(f"Probabilité de risque météorologique d'inondation : {probabilite_inondation:.1%}")

Date analysée : 2026-07-07
Probabilité de risque météorologique d'inondation : 0.0%


In [4]:
# Chargement de la table des quartiers avec indice de risque (notebook 2)
quartiers = pd.read_csv(r"C:\Users\ADS\Desktop\Projet_Inondation\quartiers_risque_libreville.csv")

print(f"Nombre de quartiers chargés : {len(quartiers)}")
quartiers[["quartier", "indice_risque"]]

Nombre de quartiers chargés : 12


,quartier,indice_risque
0,Nzeng-Ayong,0.920455
1,PK8,0.590909
2,PK6,0.000000
3,Mont-Bouët,0.863636
4,Bas-de-Gué-Gué,0.397727
5,Oloumi,0.500000
6,Plein-Ciel,0.318182
7,Awendjé,0.409091
8,Batterie 4,0.386364
9,Belle-Vue 2,0.363636


## ## Étape 3 : Combinaison météo + quartier et niveaux d'alerte

Pour chaque quartier, on combine :
- la probabilité de risque météorologique du jour (identique pour toute la
  ville, calculée à l'étape 2) ;
- l'indice de risque géospatial propre au quartier (calculé dans le
  notebook 2, basé sur l'historique des inondations et l'altitude).

Le score final permet de classer chaque quartier en trois niveaux d'alerte :
🟢 Vert (risque faible), 🟠 Orange (risque modéré), 🔴 Rouge (risque élevé).

In [5]:
# Score combiné : moyenne pondérée entre le risque météo du jour (60%)
# et le risque structurel du quartier (40%). Le risque météo pèse davantage
# car c'est lui qui déclenche réellement un épisode ponctuel ; le risque
# structurel module l'ampleur attendue selon le quartier.
quartiers["score_combine"] = (
    0.6 * probabilite_inondation + 0.4 * quartiers["indice_risque"]
)

# Attribution du niveau d'alerte selon des seuils définis
def niveau_alerte(score):
    if score >= 0.5:
        return "🔴 ROUGE"
    elif score >= 0.25:
        return "🟠 ORANGE"
    else:
        return "🟢 VERT"

quartiers["niveau_alerte"] = quartiers["score_combine"].apply(niveau_alerte)

# Affichage du tableau d'alerte, trié par score décroissant
tableau_alerte = quartiers.sort_values("score_combine", ascending=False)[
    ["quartier", "indice_risque", "score_combine", "niveau_alerte"]
]

print(f"=== Bulletin d'alerte inondation — {pd.to_datetime(date_du_jour).date()} ===")
print(f"Probabilité de risque météorologique du jour : {probabilite_inondation:.1%}\n")
print(tableau_alerte.to_string(index=False))

=== Bulletin d'alerte inondation — 2026-07-07 ===
Probabilité de risque météorologique du jour : 0.0%

      quartier  indice_risque  score_combine niveau_alerte
   Nzeng-Ayong       0.920455       0.368182      🟠 ORANGE
    Mont-Bouët       0.863636       0.345455      🟠 ORANGE
           PK8       0.590909       0.236364        🟢 VERT
        Oloumi       0.500000       0.200000        🟢 VERT
       Awendjé       0.409091       0.163636        🟢 VERT
Bas-de-Gué-Gué       0.397727       0.159091        🟢 VERT
    Batterie 4       0.386364       0.154545        🟢 VERT
   Belle-Vue 2       0.363636       0.145455        🟢 VERT
    Plein-Ciel       0.318182       0.127273        🟢 VERT
    Présidence       0.261364       0.104545        🟢 VERT
  Cité Mebiame       0.193182       0.077273        🟢 VERT
           PK6       0.000000       0.000000        🟢 VERT


## Étape 4 : Test du système avec un scénario de forte pluie (simulation)

Afin de vérifier que le système réagit correctement en conditions extrêmes,
on simule un scénario fictif de fortes précipitations cumulées, comparable
aux épisodes ayant précédé les inondations historiques documentées.

In [6]:
# Scénario fictif : fortes précipitations cumulées sur plusieurs jours
scenario_fictif = pd.DataFrame([{
    "precipitation_sum": 85.0,
    "temperature_2m_max": 27.0,
    "temperature_2m_min": 23.0,
    "temperature_2m_mean": 24.5,
    "relative_humidity_2m_mean": 95,
    "wind_speed_10m_max": 15.0,
    "surface_pressure_mean": 1008.0,
    "pluie_cumul_3j": 180.0,
    "pluie_cumul_7j": 250.0,
    "pluie_cumul_15j": 320.0,
    "humidite_moyenne_7j": 93.0,
    "amplitude_thermique": 4.0,
}])

probabilite_scenario = modele.predict_proba(scenario_fictif[colonnes_features])[0][1]
print(f"Probabilité de risque météorologique (scénario de forte pluie) : {probabilite_scenario:.1%}\n")

# Recalcul du score combiné et du niveau d'alerte avec ce scénario
quartiers["score_combine_scenario"] = (
    0.6 * probabilite_scenario + 0.4 * quartiers["indice_risque"]
)
quartiers["niveau_alerte_scenario"] = quartiers["score_combine_scenario"].apply(niveau_alerte)

tableau_scenario = quartiers.sort_values("score_combine_scenario", ascending=False)[
    ["quartier", "indice_risque", "score_combine_scenario", "niveau_alerte_scenario"]
]

print("=== Bulletin d'alerte — Scénario de forte pluie ===")
print(tableau_scenario.to_string(index=False))

Probabilité de risque météorologique (scénario de forte pluie) : 29.6%

=== Bulletin d'alerte — Scénario de forte pluie ===
      quartier  indice_risque  score_combine_scenario niveau_alerte_scenario
   Nzeng-Ayong       0.920455                0.545591                🔴 ROUGE
    Mont-Bouët       0.863636                0.522863                🔴 ROUGE
           PK8       0.590909                0.413773               🟠 ORANGE
        Oloumi       0.500000                0.377409               🟠 ORANGE
       Awendjé       0.409091                0.341045               🟠 ORANGE
Bas-de-Gué-Gué       0.397727                0.336500               🟠 ORANGE
    Batterie 4       0.386364                0.331954               🟠 ORANGE
   Belle-Vue 2       0.363636                0.322863               🟠 ORANGE
    Plein-Ciel       0.318182                0.304682               🟠 ORANGE
    Présidence       0.261364                0.281954               🟠 ORANGE
  Cité Mebiame       0.193182